In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch


tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium")


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

In [ ]:
# Let's chat for 5 lines
for step in range(5):
    # encode the new user input, add the eos_token and return a tensor in Pytorch
    new_user_input_ids = tokenizer.encode(input(">> User:") + tokenizer.eos_token, return_tensors='pt')

    # append the new user input tokens to the chat history
    bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1) if step > 0 else new_user_input_ids

    # generated a response while limiting the total chat history to 1000 tokens,
    chat_history_ids = model.generate(bot_input_ids, max_length=1000, pad_token_id=tokenizer.eos_token_id)

    # pretty print last ouput tokens from bot
    print("DialoGPT: {}".format(tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)))
2


>> User:Hello!
DialoGPT: Hello ! :D
>> User:How are you?
DialoGPT: I'm good ! How are you ?
>> User:I'm good!
DialoGPT: That's good !
>> User:Have a good day!
DialoGPT: You too ! :D
>> User:Bye! :)
DialoGPT: Bye ! :D


2

In [ ]:
import pandas as pd

In [ ]:
dados_pedidos = {
    "numero_pedido": ["12345", "67890", "11121", "22232"],
    "status": ["Shipped", "Processing", "Delivered", "Cancelled"]
}

df_status_pedidos = pd.DataFrame(dados_pedidos)
df_status_pedidos

,numero_pedido,status
0,12345,Shipped
1,67890,Processing
2,11121,Delivered
3,22232,Cancelled


In [ ]:
def verificar_status_pedido(numero_pedido):
  try:
    status = df_status_pedidos[df_status_pedidos['numero_pedido'] == numero_pedido]['status'].iloc[0]
    return f'The status of the order {numero_pedido} is: {status}'
  except:
    return 'Order not found. Please check and try again.'

In [ ]:
palavras_chave_status = ['order', 'order status', 'status of my order', 'check my order', 'track my order', 'order update']

In [ ]:
ids_historico_chat = None

while True:
  input_usuario = input('You: ')

  if input_usuario.lower() in ['exit', 'quit', 'stop']:
    print('Bot: Goodbye!')
    break

  if any(keyword in input_usuario.lower() for keyword in palavras_chave_status):
    numero_pedido = input('Could you please enter you order number?')
    resposta = verificar_status_pedido(numero_pedido)
  else:
    novo_usuario_input_ids = tokenizer.encode(input_usuario + tokenizer.eos_token, return_tensors='pt')

    if ids_historico_chat is not None:
      bot_input_ids = torch.cat([ids_historico_chat, novo_usuario_input_ids ], dim=-1)

    else:
      bot_input_ids = novo_usuario_input_ids

    ids_historico_chat = model.generate(
        bot_input_ids,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id
    )
    resposta = tokenizer.decode(ids_historico_chat[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)

  print(f'Bot: {resposta}')

You: Hello!
Bot: Hello ! :D
You: Can you check my order?
Could you please enter you order number?12345
Bot: The status of the order 12345 is: Shipped
You: Thanks! 
Bot: You're welcome ! :D
You: quit
Bot: Goodbye!


##**Interface - gradio**

In [1]:
# Importações necessárias
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import pandas as pd
import gradio as gr

# Carrega o tokenizador e o modelo
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium")

# Criando um dataframe com status de pedidos
dados_pedidos = {
    "numero_pedido": ["12345", "67890", "11121", "22232"],
    "status": ["Shipped", "Processing", "Delivered", "Cancelled"]
}
df_status_pedidos = pd.DataFrame(dados_pedidos)

# Função para verificar o status do pedido
def verificar_status_pedido(numero_pedido):
  try:
    status = df_status_pedidos[df_status_pedidos['numero_pedido'] == numero_pedido]['status'].iloc[0]

    return f'The status of your order {numero_pedido} is: {status}'

  except:
    return 'Order number not found. Please check and try again'

# Lista de palavras-chave para o status
palavras_chave_status = ['order', 'order status', 'status of my order', 'check my order', 'track my order', 'order update']

# Função para a resposta do chatbot
def responder(input_usuario, ids_historico_chat):
  if any(keyword in input_usuario.lower() for keyword in palavras_chave_status):
    return "Could you please enter your order number?", ids_historico_chat
  else:
    novo_usuario_input_ids = tokenizer.encode(input_usuario + tokenizer.eos_token, return_tensors='pt')

    if ids_historico_chat is not None:
      bot_input_ids = torch.cat([ids_historico_chat, novo_usuario_input_ids ], dim=-1)

    else:
      bot_input_ids = novo_usuario_input_ids

    ids_historico_chat = model.generate(
        bot_input_ids,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id
    )
    resposta = tokenizer.decode(ids_historico_chat[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)

  return resposta, ids_historico_chat

# Criando interface
with gr.Blocks() as app:
  chatbot = gr.Chatbot()
  msg = gr.Textbox(placeholder='Type your message here...')

  estado = gr.State(None)
  aguardando_numero_pedido = gr.State(False)

  def processar_entrada(
    input_usuario,
    historico,
    ids_historico_chat,
    aguardando_numero_pedido
  ):
      # Garante que o histórico seja uma lista
      if historico is None:
          historico = []

      if aguardando_numero_pedido:
          resposta = verificar_status_pedido(input_usuario)
          aguardando_numero_pedido = False

      else:
          resposta, ids_historico_chat = responder(
              input_usuario,
              ids_historico_chat
          )

          if resposta == "Could you please enter your order number?":
              aguardando_numero_pedido = True

      # Formato exigido pelas versões atuais do Gradio
      historico.append({
          "role": "user",
          "content": input_usuario
      })

      historico.append({
          "role": "assistant",
          "content": resposta
      })

      return (
          historico,
          ids_historico_chat,
          aguardando_numero_pedido,
          ""
      )

  msg.submit(
        fn=processar_entrada,
        inputs=[
            msg,
            chatbot,
            estado,
            aguardando_numero_pedido
        ],
        outputs=[
            chatbot,
            estado,
            aguardando_numero_pedido,
            msg
        ]
    )

  app.launch(share=True)


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/863M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ae3bd55905cc1eb144.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
